In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Configurar la sesión y cargar datos
spark = SparkSession.builder \
    .appName("Analisis Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1") \
    .getOrCreate()

# Cargar el DataFrame desde MongoDB
df = spark.read.format("mongodb").load()

In [8]:
import pandas as pd
from pymongo import MongoClient
from pyspark.sql import SparkSession

# 1. Traer los datos usando PyMongo (esto no falla)
client = MongoClient('database', 27017)
db = client['proyecto_bigdata']
coleccion = db['datos_libros']

# Convertimos los datos a una lista de diccionarios
documentos = list(coleccion.find())

# 2. Convertir a DataFrame de Spark
if documentos:
    # Quitamos el campo '_id' de Mongo porque Spark no lo maneja bien directamente
    for doc in documentos:
        doc.pop('_id', None)
    
    # Creamos la sesión de Spark normal
    spark = SparkSession.builder.appName("Analisis_Treaders").getOrCreate()
    
    # Convertimos de Lista -> Pandas -> Spark
    df_pandas = pd.DataFrame(documentos)
    df = spark.createDataFrame(df_pandas)
    
    print("¡CONEXIÓN EXITOSA!")
    df.show(5)
else:
    print("La colección está vacía. Asegúrate de haber ejecutado el Scraper (Punto 6.3) primero.")

¡CONEXIÓN EXITOSA!
+----------+------+---------+-----------+-------------------+
|      item| valor|categoria|      grupo|      fecha_captura|
+----------+------+---------+-----------+-------------------+
|Apple Inc.|189.45| Finanzas|DataTraders|2026-04-08 13:43:24|
|Tesla Inc.| 172.3| Finanzas|DataTraders|2026-04-08 13:43:24|
+----------+------+---------+-----------+-------------------+



In [9]:
from pyspark.sql.functions import col

# Filtramos para quedarnos solo con lo que tiene nombre y precio real
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print(f"Registros originales: {df.count()}")
print(f"Registros válidos: {df_filtrado.count()}")

PASO 2: Limpieza completada.
Registros originales: 2
Registros válidos: 2


In [10]:
df_filtrado.select("item", "valor").show()

+----------+------+
|      item| valor|
+----------+------+
|Apple Inc.|189.45|
|Tesla Inc.| 172.3|
+----------+------+



In [11]:
df.filter(df["valor"] > 100).show() 

+----------+------+---------+-----------+-------------------+
|      item| valor|categoria|      grupo|      fecha_captura|
+----------+------+---------+-----------+-------------------+
|Apple Inc.|189.45| Finanzas|DataTraders|2026-04-08 13:43:24|
|Tesla Inc.| 172.3| Finanzas|DataTraders|2026-04-08 13:43:24|
+----------+------+---------+-----------+-------------------+



In [12]:
df.groupBy("grupo").count().show()

+-----------+-----+
|      grupo|count|
+-----------+-----+
|DataTraders|    2|
+-----------+-----+



In [13]:
reporte_categorias = df_filtrado.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show() 

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+---------------+-------------+-------------+
|categoria|cantidad_libros|precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+---------------+-------------+-------------+
| Finanzas|              2|        180.875|        172.3|       189.45|
+---------+---------------+---------------+-------------+-------------+



In [14]:
# Ordenar por valor de forma descendente y tomar el primero
ganador = df_filtrado.orderBy(F.desc("valor")).limit(1)

print("EL PRODUCTO MÁS CARO ES:")
ganador.select("item", "valor", "categoria", "fecha_captura").show()

EL PRODUCTO MÁS CARO ES:
+----------+------+---------+-------------------+
|      item| valor|categoria|      fecha_captura|
+----------+------+---------+-------------------+
|Apple Inc.|189.45| Finanzas|2026-04-08 13:43:24|
+----------+------+---------+-------------------+



In [15]:
from pyspark.sql import functions as F

# CONFIGURACIÓN DEL GRUPO - DATATRADERS
NOMBRE_GRUPO = "DataTraders" 

# CONSULTA DE SALIDA: Resumen de integridad por Categoría
ticket_salida = df.filter(F.col("grupo") == NOMBRE_GRUPO) \
    .groupBy("categoria") \
    .agg(
        F.count("item").alias("total libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {NOMBRE_GRUPO} ---")
ticket_salida.show()

--- TICKET DE SALIDA: DataTraders ---
+---------+------------+------------+---------------------+
|categoria|total libros|precio_medio|ultima sincronizacion|
+---------+------------+------------+---------------------+
| Finanzas|           2|      180.88|  2026-04-08 13:43:24|
+---------+------------+------------+---------------------+

